In [4]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [5]:
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
import pandas as pd

books = pd.read_csv("books_cleaned.csv")

In [7]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222 On A Train Journey Home To North...
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a ma...
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...


In [8]:
books["tagged_description"]

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222 On A Train Journey Home To North...
5193    9788173031014 This book tells the tale of a ma...
5194    9788179921623 Wisdom to Create a Life of Passi...
5195    9788185300535 This collection of the timeless ...
5196    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: str

In [9]:
books["tagged_description"].to_csv("tagged_description.txt",
                                   sep = "\n",
                                   index = False,
                                   header = False)

In [10]:
raw_documents = TextLoader("tagged_description.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=8000, chunk_overlap=0, separator="\n")
documents = text_splitter.split_documents(raw_documents)

In [11]:
documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gi

In [12]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

llm = ChatOpenAI(
    model='deepseek-chat',
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url='https://api.deepseek.com',
    max_tokens=1024
)

db_books = Chroma.from_documents(
    documents,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [13]:
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k = 10)
docs

[Document(id='ab2a7619-efa8-44bd-ac47-b79c65fbb6cd', metadata={'source': 'tagged_description.txt'}, page_content='9780974607894 This vivid, sensitive tale of adolescent love follows a 16-year-old boy who falls in love with a beautiful, older woman and experiences a whirlwind of changing emotions, from exaltation and jealousy to despair and devotion. This beautifully packaged series of classic novellas includes the works of masterful writers. Inexpensive and collectible, they are the first single-volume publications of these classic tales, offering a closer look at this under-appreciated literary form and providing a fresh take on the world\'s most celebrated authors.\n"9780975323113 Gardening. Environmental Studies. Photographs by Saxon Holt. Illustrations by Richard Pembroke. This lavishly illustrated book celebrates the challenges and opportunities of gardening in Mediterranean climates, with special reference to northern California\'s San Francisco Bay Region. The core of the book i

In [14]:
books[books["isbn13"] == int(docs[0].page_content.strip('"').split()[0])]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
4294,9780974607894,0974607894,First Love,Ivan Sergeevich Turgenev,Fiction,http://books.google.com/books/content?id=VcNvD...,"This vivid, sensitive tale of adolescent love ...",2004.0,3.79,124.0,5551.0,First Love,"9780974607894 This vivid, sensitive tale of ad..."


In [15]:
def retrieve_semantic_recommendations(
        query: str,
        top_k: int = 10,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k = 50)

    books_list = []

    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]

    return books[books["isbn13"].isin(books_list)]

In [16]:
retrieve_semantic_recommendations("A book to teach children about nature")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
80,9780020867401,0020867409,"The Screwtape Letters ; With, Screwtape Propos...",Clive Staples Lewis,Christianity,NaN,One of C.S. Lewis's most imaginative creations...,1982.0,4.22,172.0,313.0,"The Screwtape Letters ; With, Screwtape Propos...",9780020867401 One of C.S. Lewis's most imagina...
106,9780060256753,0060256753,"Lafcadio, the Lion Who Shot Back",Shel Silverstein,Juvenile Fiction,http://books.google.com/books/content?id=lmeJR...,"""You don't have to shoot me,"" says the young l...",1963.0,4.15,112.0,5191.0,"Lafcadio, the Lion Who Shot Back","9780060256753 ""You don't have to shoot me,"" sa..."
146,9780060556501,0060556501,"The Lion, the Witch and the Wardrobe (picture ...",C. S. Lewis,Juvenile Fiction,http://books.google.com/books/content?id=FlSpp...,Narnia: A magical land full of wonder and exci...,2004.0,4.19,48.0,5012.0,"The Lion, the Witch and the Wardrobe (picture ...",9780060556501 Narnia: A magical land full of w...
187,9780060645892,006064589X,The Dance of the Dissident Daughter,Sue Monk Kidd,Religion,http://books.google.com/books/content?id=0FV3J...,The acclaimed spiritual memoir from the author...,2002.0,3.96,256.0,5319.0,The Dance of the Dissident Daughter,9780060645892 The acclaimed spiritual memoir f...
367,9780061124266,0061124265,Veronika Decides to Die,Paulo Coelho,Fiction,http://books.google.com/books/content?id=dlgK3...,Twenty-four-year-old Veronika seems to have ev...,2006.0,3.70,210.0,126224.0,Veronika Decides to Die: A Novel of Redemption,9780061124266 Twenty-four-year-old Veronika se...
404,9780064402453,0064402452,Racso and the Rats of NIMH,Jane Leslie Conly,Juvenile Fiction,http://books.google.com/books/content?id=MgoNv...,"‘Racso, a brash and boastful little rodent, is...",1988.0,3.76,288.0,3231.0,Racso and the Rats of NIMH,"9780064402453 ‘Racso, a brash and boastful lit..."
420,9780064408677,0064408671,The Trumpet of the Swan,E. B. White,Juvenile Fiction,http://books.google.com/books/content?id=2lybT...,"Swan Song Like the rest of his family, Louis i...",2000.0,4.07,252.0,61304.0,The Trumpet of the Swan,9780064408677 Swan Song Like the rest of his f...
434,9780064462044,0064462048,My Little House Crafts Book,Carolyn Strom Collins,Juvenile Nonfiction,http://books.google.com/books/content?id=lTzrs...,Make the same pioneer crafts that Laura did! I...,1998.0,4.05,64.0,56.0,My Little House Crafts Book: 18 Projects from ...,9780064462044 Make the same pioneer crafts tha...
442,9780067575208,006757520X,The Sense of Wonder,Rachel Carson,Nature,http://books.google.com/books/content?id=Zee5S...,"First published more than three decades ago, t...",1998.0,4.39,112.0,1160.0,The Sense of Wonder,9780067575208 First published more than three ...
449,9780071451383,0071451382,Practice Makes Perfect: Italian Verb Tenses,Paola Nanni-Tate,Foreign Language Study,http://books.google.com/books/content?id=Mbix2...,Practice makes perfect Italian verb tenses goe...,2005.0,4.00,252.0,9.0,Practice Makes Perfect: Italian Verb Tenses,9780071451383 Practice makes perfect Italian v...
